# **Correlation Study**

## Objectives

* Expand on the cancellation EDA to answer BR1:

    *TCS Hotels wants to understand cancellation patterns, trends and guest behaviour across their 2 Portuguese properties in order to identify risk factors and develop more effective cancellation defence strategies*
* Investigate statistical relationships between features and the target variable `is_canceled` using Pearson, Spearman and PPS to indentify which features carry meaningful predictive signal
* Formally validate hypotheses 1, 2 and 3 using appropriate statistical tests
* Rank features by predictive relationship with the target to inform which engineered features are required

## Inputs

* The dataset as cleaned in the [cleaning notebook](/jupyter_notebooks/04_cleaning.ipynb):

    outputs/datasets/cleaned/HotelBookingsClean.csv

## Outputs

* Relevant correlation/PPS matrices and visualisations
* Statistical test results for H1, H2 and H3
* A ranked features table saved to outputs/correlation/RankedFeatures.csv

## Additional Comments

* p-value returned as 0.0 due to floating-point underflow at this sample size; reported as p < .001


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [72]:
import os
current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [73]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

current_dir = os.getcwd()
current_dir

You set a new current directory


'/home/niall/PP4'

## Load Data

In [74]:
import pandas as pd
df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsClean.csv")
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,2023,July,27,1,0,0,2,...,C,3,No Deposit,0,0,0,Transient,0.0,0,0
1,Resort Hotel,0,737,2023,July,27,1,0,0,2,...,C,4,No Deposit,0,0,0,Transient,0.0,0,0
2,Resort Hotel,0,7,2023,July,27,1,0,1,1,...,C,0,No Deposit,0,0,0,Transient,75.0,0,0


In [ ]:
df.shape


In [ ]:
df.info()

* Since the category conversion of `is_repeated_guest`, `agent` and `company` was not preserved in the saving process, re-specify dtype

In [75]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df[col] = df[col].astype("category")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119187 entries, 0 to 119186
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype   
---  ------                          --------------   -----   
 0   hotel                           119187 non-null  object  
 1   is_canceled                     119187 non-null  int64   
 2   lead_time                       119187 non-null  int64   
 3   arrival_date_year               119187 non-null  int64   
 4   arrival_date_month              119187 non-null  object  
 5   arrival_date_week_number        119187 non-null  int64   
 6   arrival_date_day_of_month       119187 non-null  int64   
 7   stays_in_weekend_nights         119187 non-null  int64   
 8   stays_in_week_nights            119187 non-null  int64   
 9   adults                          119187 non-null  int64   
 10  children                        119187 non-null  int64   
 11  babies                          119187 non-null  int64   
 12  me

* Check the cancellation rate in unchanged following cleaning

In [ ]:
df["is_canceled"].value_counts(normalize=True)

---

## Split Feature Groups

* The dataset contains 4 distinct feature groups:
    1. The target variable (binary) ["is_canceled"]
    2. Numeric Features
    3. Categorical Features
    4. Temporal Features

In [76]:
target = ["is_canceled"]

In [77]:
numeric_features = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
]

In [78]:
categorical_features = [
    "hotel",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "is_repeated_guest",
    "agent",
    "company",
]

In [ ]:
temporal_features = [
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month"
]

In [79]:
numeric_correlation_df = pd.concat([df[target], df[numeric_features]], axis=1)
numeric_correlation_df.head()

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
0,0,342,0,0,2,0,0,0,0,3,0,0.0,0,0
1,0,737,0,0,2,0,0,0,0,4,0,0.0,0,0
2,0,7,0,1,1,0,0,0,0,0,0,75.0,0,0
3,0,13,0,1,1,0,0,0,0,0,0,75.0,0,0
4,0,14,0,2,2,0,0,0,0,0,0,98.0,0,1


---

## Pearson Correlation with Target

In [80]:
pearson_df = numeric_correlation_df.corr(method="pearson")
pearson_df

,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,previous_cancellations,previous_bookings_not_canceled,booking_changes,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
is_canceled,1.000000,0.292693,-0.001413,0.025427,0.060897,0.004662,-0.034141,0.110179,-0.057346,-0.144802,0.054338,0.048050,-0.197104,-0.234826
lead_time,0.292693,1.000000,0.085764,0.166695,0.126003,-0.037824,-0.021568,0.086082,-0.073610,0.002280,0.170101,-0.067114,-0.117681,-0.095748
stays_in_weekend_nights,-0.001413,0.085764,1.000000,0.494107,0.108900,0.045620,0.020271,-0.012757,-0.042888,0.050159,-0.054390,0.054332,-0.019104,0.073222
stays_in_week_nights,0.025427,0.166695,0.494107,1.000000,0.111423,0.043967,0.023149,-0.013966,-0.048888,0.080018,-0.002010,0.071195,-0.025667,0.068841
adults,0.060897,0.126003,0.108900,0.111423,1.000000,0.036654,0.024982,-0.007748,-0.128843,-0.047632,-0.009105,0.293216,0.018940,0.151193
children,0.004662,-0.037824,0.045620,0.043967,0.036654,1.000000,0.026727,-0.024801,-0.021114,0.050552,-0.033356,0.342263,0.057199,0.081798
babies,-0.034141,-0.021568,0.020271,0.023149,0.024982,0.026727,1.000000,-0.008007,-0.006984,0.091194,-0.011358,0.033782,0.041756,0.106927
previous_cancellations,0.110179,0.086082,-0.012757,-0.013966,-0.007748,-0.024801,-0.008007,1.000000,0.152570,-0.027264,0.005939,-0.069224,-0.018653,-0.048498
previous_bookings_not_canceled,-0.057346,-0.073610,-0.042888,-0.048888,-0.128843,-0.021114,-0.006984,0.152570,1.000000,0.011931,-0.009417,-0.075886,0.047363,0.037780
booking_changes,-0.144802,0.002280,0.050159,0.080018,-0.047632,0.050552,0.091194,-0.027264,0.011931,1.000000,-0.011918,0.026771,0.067338,0.055026


In [ ]:
import numpy as np

def heatmap_mask(df, threshold):
    mask = np.zeros_like(df, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    mask[abs(df) < threshold] = True
    return mask


* Values below ±0.1 are masked to improve readability because correlations below this level are generally considered negligible for exploratory analysis.

In [ ]:
threshold = 0.1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    pearson_df, 
    mask=(heatmap_mask(pearson_df, threshold)), 
    annot=True,
    cmap="winter",
    ax=ax)

plt.show()

---

## Spearman Correlation with Target

In [ ]:
spearman_df = numeric_correlation_df.corr(method="spearman")
spearman_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    spearman_df, 
    mask=(heatmap_mask(spearman_df, threshold)), 
    annot=True,
    cmap="winter",
    ax=ax)

plt.show()

* Full matrices are shown for completeness. Subsequent analysis focuses on the target variable.
* Compare Pearson and Spearman correlations focusing on the target variable.

In [ ]:
pearson_is_canceled = pearson_df["is_canceled"].drop(labels="is_canceled")
spearman_is_canceled = spearman_df["is_canceled"].drop(labels="is_canceled")

comparison_df = pd.concat({"Spearman": spearman_is_canceled, "Pearson": pearson_is_canceled}, axis=1)
comparison_df["Difference"] = comparison_df["Spearman"] - comparison_df["Pearson"]

comparison_df





* Prepare the data for plotting

In [ ]:
plot_comparison = comparison_df.reset_index(names="Feature")
plot_comparison = plot_comparison.drop(columns="Difference")
plot_comparison

In [ ]:
plot_comparison = plot_comparison.melt(id_vars="Feature", value_vars=["Pearson", "Spearman"], var_name="Method", value_name="Value")
plot_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle("Pearson vs Spearman Correlations")

sns.barplot(data=plot_comparison,
            x="Value",
            y="Feature",
            hue="Method")

plt.show()

* From the chart we can see that the two methods broadly agree though they diverge most on `previous_cancellations`, `previous_bookings_not_canceled` and `days_in_waiting_list`, where Spearman scores are notably larger. This is consistent with their shared heavily-skewed, zero-inflated distributions. 

---

## Predictive Power Score

In [ ]:
import ppscore as pps

pps_df = df.copy()

pps_df["is_canceled"] = pps_df["is_canceled"].astype("category")

pps_predict = pps.predictors(pps_df, y="is_canceled")
pps_predict.sort_values(by="ppscore", ascending=False)


* Take only the rows with a pscore value

In [ ]:
pps_plot_df = pps_predict[pps_predict["ppscore"] > 0].sort_values(by="ppscore", ascending=False) 
pps_plot_df


In [ ]:
fig, ax = plt.subplots()
fig.suptitle("PPS against target")
ax.set_ylabel("Feature")

sns.barplot(data=pps_plot_df,
            x="ppscore",
            y="x")

plt.show()

---

## Feature Ranking

* From the 3 correlation methods performed, we can now derive a set of relevant features
* The threshold of 0.1 has been kept for Pearson and Spearman correlations as a preferred cut-off rather than simply taking the top n features from each.

In [ ]:
pearson = pd.Series(pearson_df["is_canceled"]).drop(labels="is_canceled")
pearson = pearson.reindex(pearson.abs().sort_values(ascending=False).index)
pearson = pearson[pearson.abs() > threshold]
pearson

In [ ]:
spearman = pd.Series(spearman_df["is_canceled"]).drop(labels="is_canceled")
spearman = spearman.reindex(spearman.abs().sort_values(ascending=False).index)
spearman = spearman[spearman.abs() > threshold]
spearman

In [ ]:
pps_col = pps_plot_df.set_index("x")["ppscore"].sort_values(ascending=False)
pps_col

In [ ]:
all_features = pearson.index.union(spearman.index).union(pps_col.index)

ranked_df = pd.DataFrame(index=all_features)
ranked_df["pps"] = pps_col
ranked_df["pearson"] = pearson
ranked_df["spearman"] = spearman

# Scores are ordered primarily by pps as the only method able to assess all variables 
ranked_df.sort_values(by="pps", ascending=False)

* Expand the ranked features dataframe to include type information

In [ ]:
ranked_df = ranked_df.reset_index(names="feature")

ranked_df

In [ ]:
feature_type = []

for feature in ranked_df["feature"]:
    if feature in numeric_features:
        feature_type.append("numeric")
    elif feature in categorical_features:
        feature_type.append("categorical")

ranked_df["feature_type"] = feature_type
ranked_df

* Assess agreememnt between methods 

In [ ]:
ranked_df["n_methods"] = ranked_df.count(axis=1, numeric_only=True)
ranked_df

* Rank variables within feature type

In [ ]:
ranked_df["strongest_signal"] = ranked_df[["pps", "pearson", "spearman"]].abs().max(axis=1)

ranked_df["rank_in_type"] = (
    ranked_df.sort_values(by=["n_methods", "strongest_signal"], ascending=False)
    .groupby("feature_type")
    .cumcount() + 1
)

ranked_df.sort_values(by=["feature_type", "rank_in_type"])

* Variables are now ranked within their category type, prioritised based on the number of methods that flagged the feature, then the strength of the strongest signal

---

## Hypothesis Testing

### H1: No deposit bookings cancel more than deposit-secured bookings

* Validation method is chi-squared test on `deposit_type` vs `is_canceled`

In [ ]:
h1_df = pd.crosstab(df["deposit_type"], df["is_canceled"])
h1_df

In [ ]:
import pingouin as pg

expected, observed, stats = pg.chi2_independence(data=df, x="deposit_type", y="is_canceled")
stats


* The Cramer's V effect value of 0.48 shows that there is a statistical relationship between the 2 variables, approaching Cohen's threshold (0.5) for a large effect at this degree of freedom. [*Source: peterstatistics.com*](https://peterstatistics.com/CrashCourse/3-TwoVarUnpair/NomNom/NomNom-2c-Effect-Size.html)

In [ ]:
h1_df = pd.crosstab(df["deposit_type"], df["is_canceled"], normalize="index")
h1_df.columns = ["not_canceled", "canceled"]

h1_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
import matplotlib.ticker as mticker

h1_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.title("Cancellation Rate by Deposit Type")
plt.show()

* However, as the chart demonstrates, the relationship is counter to the expected direction with the majority of non-refund (or pre-paid) bookings cancelling.

**Conclusion**

The crosstab shows the opposite pattern to that proposed in H1: Non Refund bookings - a deposit-secured type - show a substantially higher cancellation rate than No Deposit bookings.

The chi-square test confirms this relationship is statistically significant (χ² = 27650.12, p < 0.001) with a moderate effect size (Cramer's V = 0.48).

H1 as stated is therefore **rejected** - deposit-secured bookings, specifically the Non Refund type, cancel more than no-deposit bookings.

As a cancellation predictor, the feature remains a risk factor, just not in the way hypothesised.

This may suggest a product misalignment, an overly broad deposit-type categorisation or property specific customer behaviour that the revenue management team can explore further should they wish.

### H2: Bookings with longer lead times have a higher cancellation rate than last-minute bookings

* Validation method is point-biserial correlation between `lead_time` and `is_canceled`

In [ ]:
from scipy import stats

correlation, pvalue = stats.pointbiserialr(df["is_canceled"], df["lead_time"])

h2_stats = pd.DataFrame({
    "correlation": correlation,
    "pvalue": pvalue
}, index=["lead_time_vs_is_canceled"])

h2_stats.style.format("{:.4f}")

* The Pearson's coefficient value of 0.29 shows that there is a statistical relationship between the 2 variables, approaching Cohen's threshold (0.3) for a medium effect.
* While Cohen's convention places this result at the upper limit of a 'small' effect, more recent research suggests these thresholds are too strict. Gignac and Szodorai (2016) propose 0.10, 0.20 and 0.30 as more representative benchmarks for relatively small, typical, and relatively large effects respectively — placing this result at the upper end of a typical real-world relationship. [*Source: sciencedirect.com*](https://www.sciencedirect.com/science/article/abs/pii/S0191886916308194)

In [ ]:
bins = [-np.inf, 7 ,30 ,90, np.inf]
lead_time = pd.cut(df["lead_time"], bins, labels=["Last Minute", "Short Range", "Mid Range", "Long Range"])
h2_df = df.copy()
h2_df["lead_time"] = lead_time
h2_df = pd.crosstab(h2_df["lead_time"], h2_df["is_canceled"], normalize="index")
h2_df.columns = ["not_canceled", "canceled"]
h2_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
h2_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.title("Cancellation Rate by Lead Time Band")
plt.show()

* The chart clearly displays the relationship, the longer the lead time, the more likely the booking is to cancel

**Conclusion**

The crosstab confirms the pattern proposed in H2: As lead time increases, the proportion of cancelled bookings also increases.

The point-biserial test confirms this relationship is statistically significant (r = 0.29, p < .001) — a small effect by Cohen's convention, though at the upper end of a typical real-world relationship per Gignac and Szodorai (2016).

H2 as stated is therefore **accepted** - longer range bookings cancel more than last-minute.

As a cancellation predictor, the feature is a risk factor.

### H3: Bookings made through the Online TA market segment have a higher cancellation rate than bookings made through the Direct market segment

* Validation method is chi-squared test on `market_segment` "Direct" and "Online TA" vs `is_canceled`

In [ ]:
features = ["Online TA", "Direct"]
ota_direct = df[df["market_segment"].isin(features)]
ota_direct

h3_df = pd.crosstab(ota_direct["market_segment"], ota_direct["is_canceled"], normalize="index")
h3_df.columns = ["not_canceled", "canceled"]

h3_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

In [ ]:
expected, observed, stats = pg.chi2_independence(data=ota_direct, x="market_segment", y="is_canceled")
stats

* The Cramer's V effect value of 0.18 shows that there is a statistical relationship between the 2 variables, a small effect at this degree of freedom as per Cohen's convention. [*Source: peterstatistics.com*](https://peterstatistics.com/CrashCourse/3-TwoVarUnpair/NomNom/NomNom-2c-Effect-Size.html)

In [ ]:
h3_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.axhline(y=df["is_canceled"].mean(), linestyle="--", color="black", label="Overall cancellation rate")
plt.legend()
plt.title("Cancellation Rate Direct and Online TA")
plt.show()

* The chart shows that the cancellation rate for Online TA is roughly the same as the overall cancellation rate, while the cancellation rate for direct is considerably below average.

In [ ]:
sns.countplot(data=ota_direct, x="market_segment", hue="is_canceled")
plt.show()

* This chart shows that while OTA bookings cancel at a rate roughly on par with the overall average, their much larger booking volume means they account for a disproportionately high number of cancelled reservations

**Conclusion**

The crosstab confirms the pattern proposed in H3: Online TA bookings have a higher cancellation rate than Direct bookings.

The chi-square test confirms this relationship is statistically significant (χ² = 2146.33, p < 0.001) with a small effect size (Cramer's V = 0.18).

H3 as stated is therefore **accepted** - Online TA bookings have a higher cancelltion rate than Direct bookings.

As a predictive feature, market segment (Online TA vs Direct) remains a useful indicator of cancellation risk.

Although the Online TA aligns with the dataset average, bookings from this segment represent a much larger booking volume. Consequently, they contribute a disproportionately large share of all cancelled reservations, which helps explain why they are often perceived operationally as a major source of cancellations.

* Create table of results

In [ ]:
hypothesis_results = pd.DataFrame({
    "hypothesis": ["H1", "H2", "H3"],
    "test": ["Chi-square", "Point-biserial", "Chi-Square"],
    "statistic": ["χ² = 27650.12", "r_pb = 0.29", "χ² = 2146.33"],
    "p_value": ["p < 0.001", "p < 0.001", "p < 0.001"],
    "effect_size": ["Cramer's V = 0.48", "r_pb = 0.29", "Cramer's V = 0.18"],
    "outcome": ["Rejected", "Accepted", "Accepted"]
})

hypothesis_results

---

## BR1 Conclusions

This notebook, alongside the exploratory work in the Cancellation EDA, addresses BR1 by formally identifying and validating cancellation risk factors across TCS Hotels' two Portuguese properties.

Three confirmed risk factors emerge from hypothesis testing:

* **Deposit type** is the strongest identified risk factor, though the direction runs counter to initial expectations. Non Refund bookings — not No Deposit bookings — show a substantially higher cancellation rate. This is the single strongest categorical signal in the dataset.
* **Lead time** is a confirmed risk factor: cancellation likelihood rises consistently as lead time increases..
* **Market segment** (Online TA vs Direct) is a confirmed, smaller-effect risk factor. While its cancellation rate sits close to the property-wide average, the segment's disproportionate share of total bookings means it accounts for a large volume of absolute cancellations — a factor operationally significant even where the underlying rate is unremarkable.

Together, these findings indicate that cancellation risk at TCS Hotels is driven less by *when* a guest books relative to arrival alone, and more by the *type of booking commitment* involved — deposit terms and booking channel carry stronger signal than volume or seasonality alone. This reframes the client's risk factors away from a simple "early bookers are safer" assumption toward booking-product characteristics as the primary driver of cancellation behaviour, informing the defence strategies BR1 calls for: targeted intervention around Non Refund and Online TA bookings is likely to yield more impact than blanket lead-time-based policies.

---

## Feature Engineering Recommendations

**Guest Composition**
* Investigate whether combining `adults`, `children` and `babies` into one total guest count provides a stronger predictive signal
* Assess whether an `is_family` binary flag is of more predictive value than guest counts alone

**Stay Characteristics**
* Evaluate whether combining `stays_in_week_nights` and `stays_in_weekend_nights` into a total length of stay (LOS) improves predictive performance
* Investigate whether arrival day of the week is a factor such as `arrival_is_weekend` or if weekend ratio is an indicator

**Temporal Variables**
* Assess whether cyclical encoding of week number and month better represent seasonality
* Evaluate whether broader seasonal features outperform individual calendar components

**Booking Behaviour**
* Explore whether lead time benefits from transformation or categorisation
* Consider whether previous history can be better represented through derived behavioural features

**Long Tail Compression**
* Investigate whether compiling long-tail categorical features into top-n + other produces a more significant result

**Validation**
* Re-run Pearson, Spearman and PPS on all engineered features
* Compare engineered features with the raw variables they replace to inform the decision to keep or revert

---

# Outputs

In [ ]:
import os
try:
  os.makedirs(name='outputs/correlation')
except Exception as e:
  print(e)


* Save RankedFeatures for use in [feature engineering](/jupyter_notebooks/06_feature_engineering.ipynb)
* Save HypothesisResults for use in the dashboard

In [ ]:
ranked_df.to_csv("outputs/correlation/RankedFeatures.csv")
hypothesis_results.to_csv("outputs/correlation/HypothesisResults.csv")